# Step 1 — Compare DB vs Elasticsearch

Run this notebook first. It compares counts between PostgreSQL and Elasticsearch,
saves report files, and tells you which scenario notebook to open next.

| Result | Next step |
|--------|-----------|
| Records missing from DB | Open `2_missing_in_db.ipynb` |
| Records missing from ES | Open `3_missing_in_elastic.ipynb` |

In [ ]:
import json, os, sys, warnings
from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, FileLink, HTML
warnings.filterwarnings('ignore')

repo_root = Path('.').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pipeline import compare as _cmp

with open('campaign_config.json') as f:
    _raw = json.load(f)
_GLOBALS  = _raw.get('_globals', {})
CAMPAIGNS = {k: v for k, v in _raw.items() if k != '_globals'}

with open('metrics_config.json') as f:
    METRICS = json.load(f)

def get_cfg(key):
    return {**_GLOBALS, **CAMPAIGNS[key]}

def _dl(path, label='Download'):
    if path and os.path.exists(str(path)):
        display(FileLink(str(path), result_html_prefix=f'\u2b07  {label}: '))

print('Setup complete.')
print('Campaigns:', list(CAMPAIGNS.keys()))
print('Metrics  :', list(METRICS.keys()))

In [ ]:
_W = {'description_width': '110px'}
_M = widgets.Layout(width='320px')

w_campaign = widgets.Dropdown(
    options=[(v['label'], k) for k, v in CAMPAIGNS.items()],
    description='Campaign:', style=_W, layout=_M
)

def _cfg_html(key):
    c = get_cfg(key)
    def s(v):
        color = 'green' if v else 'red'
        mark  = '✓ set' if v else '✗ NOT SET'
        return f"<span style='color:{color}'>{mark}</span>"
    rows = (
        f"<b>DB:</b> {c['db_host']} / {c['db_name']} &nbsp; user: {c['db_user']}<br>"
        f"<b>ES:</b> {c['es_base_url']} &nbsp; user: {c['es_username']} &nbsp; pass: {s(c.get('es_password'))}<br>"
        f"<b>Auth Token:</b> {s(c.get('auth_token'))}<br>"
        f"<b>Project Type ID:</b> {s(c.get('projectTypeId'))} &nbsp; "
        f"<b>Province:</b> {s(c.get('province'))}"
    )
    return (
        "<div style='font-family:monospace;background:#f9f9f9;padding:10px 14px;"
        "border-radius:4px;border:1px solid #ddd;line-height:2'>"
        + rows + "</div>"
    )

w_info = widgets.HTML(value=_cfg_html(w_campaign.value))
w_campaign.observe(lambda ch: setattr(w_info, 'value', _cfg_html(ch['new'])), names='value')

w_metric = widgets.Dropdown(options=list(METRICS.keys()), description='Metric:', style=_W, layout=widgets.Layout(width='420px'))
w_outdir = widgets.Text(value='output', description='Output dir:', style=_W, layout=_M)

display(widgets.HTML('<h3>Campaign &amp; Credentials</h3>'))
display(w_campaign, w_info)
display(widgets.HBox([w_metric, w_outdir]))

In [ ]:
btn = widgets.Button(description='Run Compare', button_style='primary', icon='search',
                     layout=widgets.Layout(width='200px'))
out = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='8px 0'))

def _run(b):
    btn.disabled = True
    with out:
        out.clear_output()
        try:
            key = w_campaign.value
            cfg = get_cfg(key)
            missing_cfg = [k for k in ['es_password', 'auth_token'] if not cfg.get(k)]
            if missing_cfg:
                print(f'ERROR: Fill in campaign_config.json [{key}]: {", ".join(missing_cfg)}'); return

            output_dir = os.path.join(w_outdir.value.strip() or 'output', key)
            os.makedirs(output_dir, exist_ok=True)

            db_config = {
                'host': cfg['db_host'], 'port': cfg['db_port'], 'database': cfg['db_name'],
                'user': cfg['db_user'], 'password': cfg['db_pass'],
                'sslmode': 'require', 'connect_timeout': 30,
            }

            print(f"Comparing: {cfg['label']} / {w_metric.value}")

            result = _cmp.run_compare(
                campaign_cfg = cfg,
                metric_cfg   = METRICS[w_metric.value],
                db_config    = db_config,
                es_username  = cfg['es_username'],
                es_password  = cfg['es_password'],
                output_dir   = output_dir,
            )

            r      = result
            miss_a = r['missing_in_db']
            miss_b = r['missing_in_elastic']

            display(HTML(
                f"<table style='border-collapse:collapse;font-family:monospace;font-size:14px'>"
                f"<tr><th style='text-align:left;padding:5px 20px 5px 5px;border-bottom:1px solid #ccc'>Metric</th>"
                f"<th style='text-align:right;padding:5px;border-bottom:1px solid #ccc'>Count</th></tr>"
                f"<tr><td style='padding:4px 20px 4px 5px'>Total in DB</td><td style='text-align:right'>{r['db_ids']:,}</td></tr>"
                f"<tr><td style='padding:4px 20px'>Total in ES</td><td style='text-align:right'>{r['es_ids']:,}</td></tr>"
                f"<tr><td style='padding:4px 20px'>Matched</td><td style='text-align:right'>{r['matched']:,}</td></tr>"
                f"<tr style='background:#{'fff3f3' if miss_a else 'f3fff3'}'>"
                f"<td style='padding:5px 20px'><b>Missing in DB (ES has it) → 2_missing_in_db.ipynb</b></td>"
                f"<td style='text-align:right;color:{'red' if miss_a else 'green'}'><b>{miss_a:,}</b></td></tr>"
                f"<tr style='background:#{'fff3f3' if miss_b else 'f3fff3'}'>"
                f"<td style='padding:5px 20px'><b>Missing in ES (DB has it) → 3_missing_in_elastic.ipynb</b></td>"
                f"<td style='text-align:right;color:{'red' if miss_b else 'green'}'><b>{miss_b:,}</b></td></tr>"
                f"</table>"
            ))

            session = {
                'campaign_key': key, 'metric': w_metric.value,
                'output_dir': output_dir, 'ts': result['ts'],
                'missing_in_db': miss_a, 'missing_in_elastic': miss_b,
            }
            session_path = os.path.join(output_dir, 'session.json')
            with open(session_path, 'w') as f:
                json.dump(session, f, indent=2)

            display(HTML('<br><b>Download reports:</b>'))
            for fname in sorted(os.listdir(output_dir)):
                if fname.endswith(('.csv', '.json')):
                    _dl(os.path.join(output_dir, fname), fname)

            display(HTML(
                f"<br><span style='color:green'>✓ Session saved → <code>{session_path}</code><br>"
                f"Point 2_missing_in_db.ipynb or 3_missing_in_elastic.ipynb to: <code>{output_dir}</code></span>"
            ))
        except Exception:
            import traceback; traceback.print_exc()
        finally:
            btn.disabled = False

btn.on_click(_run)
display(widgets.HTML('<h3>Run Compare</h3>'))
display(btn, out)